# Sequence-to-sequence translation with attention, from scratch

This is a teaching notebook. We build an attention translator one component at a
time, deriving each piece before we run it: a bidirectional encoder, Bahdanau
(additive) attention, and an attending decoder. We train it on a controlled
task, converting human-written dates into ISO 8601, and then read the learned
attention as a heatmap to see the alignment the model discovered.

The task is deliberately small so the whole pipeline runs on a CPU in a couple of
minutes and the attention is interpretable. The companion note `docs/attention.md`
gives the full derivation; here we move quickly and let the code carry the story.

## 1. The task

The source language is dates written the way people write them, in several
formats. The target language is the single canonical ISO form `YYYY-MM-DD`. This
is a genuine sequence-to-sequence alignment problem: the model must copy digits,
map month names to two-digit numbers, and reorder the day, month, and year
fields. Everything is generated deterministically from a seed, so there is no
download and the run is reproducible.

In [1]:
from nmt import make_date_dataset, set_seed

set_seed(0)
data = make_date_dataset(n=6000, seed=0)
for src, tgt in list(zip(data.sources, data.targets))[:8]:
    print(f"{src:<24} -> {tgt}")
print()
print("source vocab size:", len(data.src_vocab))
print("target vocab size:", len(data.tgt_vocab))

August 15, 2025          -> 2025-08-15
3 January 1990           -> 1990-01-03
19.10.1981               -> 1981-10-19
Aug 28, 2002             -> 2002-08-28
16.07.2011               -> 2011-07-16
19 October 1988          -> 1988-10-19
16 November 1995         -> 1995-11-16
September 24, 2019       -> 2019-09-24

source vocab size: 46
target vocab size: 15


## 2. The encoder

The encoder reads the source characters with a bidirectional GRU and returns one
hidden state per source position. For a source of length `S` we get states
`h_1, ..., h_S`, each of dimension `2H` because the forward and backward
directions are concatenated. These per-position states are what attention will
later weigh. The encoder also produces an initial decoder state by combining the
final forward and backward states through a small linear bridge and a `tanh`.

Note the shapes: the encoder outputs are `(batch, S, 2H)`, not a single summary
vector. Keeping every position is the whole point; it is what makes attention
possible.

In [2]:
import torch
from nmt import Seq2Seq
from nmt.data import tensorize_batch

model = Seq2Seq(
    len(data.src_vocab), len(data.tgt_vocab),
    embed_dim=48, hidden_dim=96,
    src_pad=data.src_vocab.pad_id, tgt_pad=data.tgt_vocab.pad_id,
)

# Encode one small batch and inspect the shapes.
src_ids, _, tgt_ids, _ = tensorize_batch(
    data.sources[:4], data.targets[:4], data.src_vocab, data.tgt_vocab
)
enc_outputs, dec_init = model.encoder(src_ids)
print("source ids       :", tuple(src_ids.shape), "(batch, S)")
print("encoder outputs  :", tuple(enc_outputs.shape), "(batch, S, 2H)")
print("initial dec state:", tuple(dec_init.shape), "(batch, H)")

source ids       : (4, 15) (batch, S)
encoder outputs  : (4, 15, 192) (batch, S, 2H)
initial dec state: (4, 96) (batch, H)


## 3. Bahdanau attention

At each decoding step the decoder holds a state `s`. Additive attention compares
that state against every encoder state `h_j` with a small feed-forward network,
producing one score per source position:

$$e_j = v^\top \tanh(W_{enc}\, h_j + W_{dec}\, s)$$

The scores at padded positions are set to negative infinity, then a softmax turns
them into weights `alpha_j` that sum to one. The context vector is the weighted
average of the encoder states, `c = sum_j alpha_j h_j`, a focused re-reading of
the source for this step.

Let us run the attention module on the encoder outputs with the initial decoder
state and confirm the weights form a proper distribution over source positions.

In [3]:
mask = src_ids != model.src_pad
context, weights = model.decoder.attention(dec_init, enc_outputs, mask)
print("context vector   :", tuple(context.shape), "(batch, 2H)")
print("attention weights:", tuple(weights.shape), "(batch, S)")
print("row sums (should be ~1):", [round(x, 4) for x in weights.sum(dim=-1).tolist()])

context vector   : (4, 192) (batch, 2H)
attention weights: (4, 15) (batch, S)
row sums (should be ~1): [1.0, 1.0, 1.0, 1.0]


## 4. The decoder

The decoder is a GRU cell that runs one step at a time. At step `t` it takes the
embedding of the previous output character concatenated with the context vector
`c_t`, updates its state, and maps the state-and-context pair to logits over the
target vocabulary. During training we feed the true previous character (teacher
forcing); at inference we feed the model's own previous prediction (greedy
decoding). Both paths share the exact same `Decoder.step`, so what we train is
what we run.

## 5. Training

We train with teacher forcing and cross-entropy, ignoring padded target
positions. On this task the loss falls quickly. We keep the notebook run short;
the full run in `scripts/train.py` uses more data and epochs and reaches a
perfect score.

In [4]:
from nmt import Trainer

trainer = Trainer(model, tgt_pad=data.tgt_vocab.pad_id, lr=1e-3)
trainer.fit(data, epochs=10, batch_size=128, verbose=True)

epoch   1  loss=1.8245


epoch   2  loss=0.6689


epoch   3  loss=0.2286


epoch   4  loss=0.0783


epoch   5  loss=0.0203


...

epoch   7  loss=0.0035


epoch   8  loss=0.0024


epoch   9  loss=0.0018


epoch  10  loss=0.0014


Trainer(model=Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(46, 48, padding_idx=0)
    (rnn): GRU(48, 96, batch_first=True, bidirectional=True)
    (bridge): Linear(in_features=192, out_features=96, bias=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(15, 48, padding_idx=0)
    (attention): BahdanauAttention(
      (W_enc): Linear(in_features=192, out_features=96, bias=False)
      (W_dec): Linear(in_features=96, out_features=96, bias=False)
      (v): Linear(in_features=96, out_features=1, bias=False)
    )
    (rnn): GRUCell(240, 96)
    (out): Linear(in_features=288, out_features=15, bias=True)
  )
), tgt_pad=0, lr=0.001, device='cpu', history={'loss': [1.8244942874908447, 0.6689048619270325, 0.22855870099862416, 0.07833813352386157, 0.020333389090994993, 0.006385656028985977, 0.0034875517344723143, 0.0023622014997527, 0.0017504272647202014, 0.0013659929055720567]})

## 6. Evaluation

We measure exact-match accuracy, the fraction of dates normalized perfectly,
character for character, on a fresh set of dates drawn from a different seed.
Greedy decoding runs the decoder autoregressively until it emits the end token.

In [5]:
from nmt import translate, exact_match_accuracy

held_out = make_date_dataset(n=800, seed=99)
preds = translate(model, held_out.sources, data.src_vocab, data.tgt_vocab)
acc = exact_match_accuracy(preds, held_out.targets)
print(f"exact-match accuracy on 800 fresh dates: {acc:.4f}\n")
for src, pred, ref in list(zip(held_out.sources, preds, held_out.targets))[:8]:
    flag = "ok" if pred == ref else "X"
    print(f"[{flag}] {src:<26} -> {pred}  (gold {ref})")

exact-match accuracy on 800 fresh dates: 1.0000

[ok] 07/22/2032                 -> 2032-07-22  (gold 2032-07-22)
[ok] 27.07.1981                 -> 1981-07-27  (gold 1981-07-27)
[ok] 08/09/2032                 -> 2032-08-09  (gold 2032-08-09)
[ok] 04/27/2012                 -> 2012-04-27  (gold 2012-04-27)
[ok] 06/03/1992                 -> 1992-06-03  (gold 1992-06-03)
[ok] December 9, 1992           -> 1992-12-09  (gold 1992-12-09)
[ok] 10th of April 2009         -> 2009-04-10  (gold 2009-04-10)
[ok] 26th of April 2014         -> 2014-04-26  (gold 2014-04-26)


## 7. The hero: reading the attention

Now the payoff. We decode one example and plot the attention matrix. Each row is
one generated output character and each column is one source character; a bright
cell means the decoder attended to that source character when producing that
output. If attention has learned the alignment, the year digits should attend to
the source year, the month digits to the month name, and the day digits to the
source day.

In [6]:
from nmt import plot_attention

fig = plot_attention(model, "28th of August 2012", data.src_vocab, data.tgt_vocab)
fig

<Figure size 855x420 with 2 Axes>

The pattern is sharp and interpretable: each output field attends to exactly the
source characters it copies or maps from. That is the mechanism working as
designed.

## 8. Honest framing

The exact-match score on the full run is 1.0. That is not a machine-translation
benchmark and should not be read as one. The date task is synthetic, finite in
structure, and fully learnable on a CPU, so a correctly built attention model is
expected to solve it completely. The perfect score is controlled validation: it
confirms the encoder, attention, decoder, masking, and greedy decoder are wired
correctly, and the heatmap confirms the model succeeds by aligning source and
target rather than by a shortcut. The same architecture, unchanged, applies to a
natural-language corpus such as Multi30k, where scores would be far lower and
beam search and subword tokenization would start to matter.